# 02 Data Preprocessing

This notebook prepares the raw Amazon reviews for representation learning by:
- Cleaning the text (removing HTML, special characters)
- Handling missing values
- Deduplication
- Saving to `data/processed/`

In [ ]:
import pandas as pd
import re
import os
from glob import glob


## 1. Load Raw Data

In [ ]:
raw_files = glob("../data/raw/*.jsonl")
if not raw_files:
    raise FileNotFoundError("No raw data found. Run the download script first.")

latest_file = max(raw_files, key=os.path.getctime)
print(f"Processing: {latest_file}")

df = pd.read_json(latest_file, lines=True)
print(f"Initial shape: {df.shape}")

## 2. Cleaning Functions

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Note: For representation learning (like SBERT/BGE), 
    # we often want to keep punctuation and case.
    return text

def preprocess_pipeline(df):
    print("Cleaning text...")
    df['cleaned_text'] = df['text'].apply(clean_text)
    
    # Remove reviews with empty text
    print("Removing empty reviews...")
    df = df[df['cleaned_text'] != ""].copy()
    
    # Deduplication
    print("Deduplicating...")
    df = df.drop_duplicates(subset=['user_id', 'parent_asin', 'cleaned_text'])
    
    return df

## 3. Run Pipeline

In [ ]:
df_processed = preprocess_pipeline(df)
print(f"Processed shape: {df_processed.shape}")
print(f"Dropped {len(df) - len(df_processed)} rows.")

In [ ]:
print("Sample before/after cleaning:")
for i in range(min(3, len(df_processed))):
    print(f"Original: {df_processed['text'].iloc[i][:100]}...")
    print(f"Cleaned:  {df_processed['cleaned_text'].iloc[i][:100]}...")
    print("-" * 30)

## 4. Save Processed Data

In [ ]:
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "cleaned_reviews.parquet")
print(f"Saving to {output_path}...")

# We use parquet for better compression and speed
df_processed.to_parquet(output_path, index=False)
print("Done!")